# 🍄 Gljive Hrvatske – vlastiti model za prepoznavanje

Ova bilježnica sama skine otvorene fotografije gljiva s **iNaturalista** i **Wikimedia Commonsa** (samo licence **CC0**, javno dobro i **CC BY**, koje dopuštaju i komercijalnu upotrebu), istrenira model za vrste iz atlasa i spremi ga u oblik koji aplikacija koristi.

**Kako pokrenuti**
1. Izbornik **Runtime → Change runtime type → T4 GPU → Save**.
2. Izbornik **Runtime → Run all**.
3. Kad pita, dopusti pristup **Google Driveu** (tamo se sve sprema, u mapu `GljiveModel`).
4. Pričekaj. Prvi put traje otprilike **3–5 sati**; sljedeće verzije brže jer su popisi i fotografije već na Driveu (skidanje popisa ~1 h, fotografija ~30 min, treniranje ~2–3 h).
5. Ako se Colab prekine ili zatvori: ponovno **Runtime → Run all**. Nastavlja gdje je stao.
6. Na kraju se u mapi `GljiveModel` na Driveu pojavi **`GljiveModel_v5.zip`**. Pošalji ga Claudeu i on ga ubaci u aplikaciju i usporedi s danskim modelom.

Licenca osnovnog modela: `efficientnet_b2.ra_in1k` iz biblioteke timm (Apache 2.0). Testne fotografije iz atlasa izbacuju se iz učenja. Popis autora fotografija sprema se u `zasluge.csv`.

In [ ]:
# ===== POSTAVKE (ne treba mijenjati) =====
MODEL_NAME   = "efficientnet_b2.ra_in1k"  # osnovni model (Apache 2.0), veći i točniji od b0
IMG          = 256
MAX_PO_VRSTI = 500        # najviše fotografija po vrsti
MIN_PO_VRSTI = 20         # vrste s manje fotografija se preskaču
FOTO_PO_NALAZU = 2        # najviše fotografija s istog nalaza
LICENCE      = ["cc0", "cc-by"]   # samo licence koje dopuštaju i komercijalnu upotrebu
EPOHE        = 24
BATCH        = 96
LR           = 8e-4
VERZIJA      = 5
PRETRAINED   = True
IMG_FINAL    = 288        # zadnje epohe uče na većoj slici; model u aplikaciji radi na 288 px
FINAL_EP     = 4
COMMONS      = True       # dodatne fotografije s Wikimedia Commonsa (CC0, javno dobro, CC BY)
COMMONS_MAX  = 200        # najviše fotografija s Commonsa po vrsti
API_PAUZA    = 1.0        # sekunde između upita iNaturalistu (njihovo pravilo: najviše ~1 u sekundi)

In [ ]:
%pip install -q timm onnx onnxruntime

In [ ]:
import os, io, json, time, csv, math, random, hashlib, tarfile, zipfile, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from PIL import Image
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT  = Path('/content/drive/MyDrive/GljiveModel')
    WORK = Path('/content/work')
except ImportError:  # Kaggle ili vlastito računalo
    OUT  = Path(os.environ.get('GH_OUT', '/kaggle/working/GljiveModel' if Path('/kaggle').exists() else 'GljiveModel'))
    WORK = Path(os.environ.get('GH_WORK', str(OUT.parent / 'gh_work')))
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
(OUT / 'popis').mkdir(exist_ok=True)
SLIKE = WORK / 'slike'; SLIKE.mkdir(exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Spremam u:', OUT, '| uređaj:', DEV)
if DEV == 'cpu':
    print('⚠️ Nema GPU-a! Runtime → Change runtime type → T4 GPU, pa ponovno Run all. Bez GPU-a treniranje traje danima.')

In [ ]:
VRSTE = ["Boletus edulis", "Boletus reticulatus", "Boletus aereus", "Tylopilus felleus", "Rubroboletus satanas", "Cantharellus cibarius", "Hygrophoropsis aurantiaca", "Omphalotus olearius", "Macrolepiota procera", "Lepiota spp.", "Amanita phalloides", "Amanita virosa", "Amanita muscaria", "Amanita pantherina", "Amanita caesarea", "Amanita rubescens", "Morchella esculenta", "Gyromitra esculenta", "Pleurotus ostreatus", "Armillaria mellea", "Hypholoma fasciculare", "Galerina marginata", "Agaricus campestris", "Agaricus xanthodermus", "Russula virescens", "Russula emetica", "Lactarius deliciosus", "Craterellus cornucopioides", "Calocybe gambosa", "Inosperma erubescens", "Entoloma sinuatum", "Coprinus comatus", "Cortinarius orellanus", "Tuber magnatum", "Boletus pinophilus", "Butyriboletus regius", "Butyriboletus pseudoregius", "Butyriboletus appendiculatus", "Caloboletus calopus", "Neoboletus erythropus", "Suillellus luridus", "Suillellus queletii", "Imleria badia", "Xerocomus subtomentosus", "Xerocomellus chrysenteron", "Hortiboletus rubellus", "Gyroporus castaneus", "Gyroporus cyanescens", "Leccinum scabrum", "Leccinum pseudoscabrum", "Leccinum aurantiacum", "Leccinellum crocipodium", "Suillus luteus", "Suillus granulatus", "Suillus bovinus", "Suillus grevillei", "Suillus variegatus", "Chalciporus piperatus", "Strobilomyces strobilaceus", "Gomphidius glutinosus", "Craterellus tubaeformis", "Craterellus lutescens", "Hydnum repandum", "Sarcodon imbricatus", "Chlorophyllum rhacodes", "Macrolepiota mastoidea", "Lepiota cristata", "Lepiota clypeolaria", "Echinoderma asperum", "Agaricus augustus", "Agaricus bitorquis", "Agaricus sylvaticus", "Amanita verna", "Amanita citrina", "Amanita gemmata", "Amanita vaginata", "Coprinopsis atramentaria", "Coprinopsis picacea", "Clitocybe nebularis", "Infundibulicybe geotropa", "Infundibulicybe gibba", "Clitocybe odora", "Clitocybe rivulosa", "Clitopilus prunulus", "Lepista nuda", "Lepista glaucocana", "Lepista personata", "Marasmius oreades", "Tricholoma portentosum", "Tricholoma columbetta", "Tricholoma pardinum", "Tricholoma saponaceum", "Tricholoma sejunctum", "Tricholoma sciodes", "Tricholomopsis rutilans", "Melanoleuca melaleuca", "Lyophyllum decastes", "Lyophyllum fumosum", "Lyophyllum loricatum", "Leucocybe connata", "Laetiporus sulphureus", "Fistulina hepatica", "Grifola frondosa", "Meripilus giganteus", "Polyporus umbellatus", "Albatrellus cristatus", "Albatrellus pes-caprae", "Hapalopilus rutilans", "Auricularia auricula-judae", "Flammulina velutipes", "Hypholoma lateritium", "Psathyrella piluliformis", "Cyclocybe cylindracea", "Agrocybe praecox", "Desarmillaria tabescens", "Ganoderma lucidum", "Fomitopsis betulina", "Trametes versicolor", "Lentinus tigrinus", "Panus conchatus", "Oudemansiella mucida", "Hymenopellis radicata", "Gymnopus fusipes", "Rhodocollybia butyracea", "Megacollybia platyphylla", "Laccaria laccata", "Mycena pura", "Mycena inclinata", "Pluteus cervinus", "Volvariella bombycina", "Volvopluteus gloiocephalus", "Lactarius sanguifluus", "Lactarius deterrimus", "Lactarius semisanguifluus", "Lactarius chrysorrheus", "Lactarius torminosus", "Lactarius blennius", "Lactifluus piperatus", "Lactifluus vellereus", "Lactifluus volemus", "Russula cyanoxantha", "Russula grisea", "Russula vesca", "Russula heterophila", "Russula aurea", "Russula romellii", "Russula delica", "Russula nigricans", "Russula lepida", "Russula olivacea", "Cortinarius caperatus", "Cortinarius praestans", "Cortinarius bolaris", "Cortinarius semisanguineus", "Cortinarius trivialis", "Cortinarius rubellus", "Leucocortinarius bulbiger", "Pseudosperma rimosum", "Hebeloma sinapizans", "Paxillus involutus", "Tapinella atrotomentosa", "Entoloma clypeatum", "Entoloma sepium", "Hygrophorus marzuolus", "Hygrophorus russula", "Hygrophorus nemoreus", "Hygrophorus poetarum", "Cuphophyllus pratensis", "Hygrocybe punicea", "Stropharia aeruginosa", "Stropharia rugosoannulata", "Ramaria flava", "Ramaria botrytis", "Ramaria formosa", "Lycoperdon perlatum", "Apioperdon pyriforme", "Calvatia gigantea", "Scleroderma citrinum", "Phallus impudicus", "Clathrus ruber", "Morchella elata", "Pleurotus eryngii", "Tuber aestivum", "Fomitopsis pinicola", "Fomes fomentarius", "Coprinellus micaceus", "Stereum hirsutum", "Mycena galericulata", "Cuphophyllus virgineus", "Hygrocybe miniata", "Ganoderma applanatum", "Gymnopilus penetrans", "Armillaria lutea", "Phaeolepiota aurea", "Trametes hirsuta", "Trametes gibbosa", "Pseudocraterellus undulatus", "Schizophyllum commune", "Kuehneromyces mutabilis", "Laccaria amethystina", "Hygrocybe ceracea", "Xerocomellus pruinatus", "Hygrocybe conica", "Pholiota squarrosa", "Gliophorus psittacinus", "Phaeolus schweinitzii", "Tricholoma scalpturatum", "Russula ochroleuca", "Paralepista flaccida", "Ramaria stricta", "Tricholoma terreum", "Russula adusta", "Mycetinis alliaceus", "Lacrymaria lacrymabunda", "Ganoderma pfeifferi", "Hygrocybe coccinea", "Amanita fulva", "Chlorophyllum olivieri", "Coprinellus disseminatus", "Inocybe geophylla", "Mycena rosea", "Pseudoclitocybe cyathiformis", "Armillaria ostoyae", "Boletus ferrugineus", "Stropharia cyanea", "Amanita excelsa", "Mycena crocata", "Mycena haematopus", "Clavulina rugosa", "Hygrocybe acutoconica", "Clavulina coralloides", "Geastrum michelianum", "Clitocybe fragrans", "Laccaria proxima", "Mycena epipterygia", "Trametes ochracea", "Polyporus tuberaster", "Lactarius quietus", "Hygrocybe chlorophana", "Gymnopus dryophilus", "Caloboletus radicans", "Gymnopus confluens", "Agaricus impudicus", "Protostropharia semiglobata", "Lycoperdon nigrescens", "Tricholoma sulphureum", "Coprinopsis lagopus", "Panaeolus papilionaceus", "Thelephora terrestris", "Pseudohydnum gelatinosum", "Cortinarius elatior", "Marasmius rotula", "Hypholoma capnoides", "Pleurotus pulmonarius", "Pholiota gummosa", "Panaeolina foenisecii", "Hericium coralloides", "Rhodocollybia maculata", "Gliophorus laetus", "Hygrocybe cantharellus", "Lactarius necator", "Lycoperdon excipuliforme", "Strobilurus esculentus", "Mycena galopus", "Leucoagaricus nympharum", "Mycena vitilis", "Hortiboletus bubalinus", "Lactarius subdulcis", "Mycena tintinnabulum", "Tricholoma argyraceum", "Pluteus salicinus", "Lactarius pubescens", "Lactarius rufus", "Cortinarius largus", "Cortinarius malicorius", "Cortinarius torvus", "Cortinarius anserinus", "Cortinarius flexipes", "Mycena leptocephala", "Cortinarius croceus", "Agaricus sylvicola", "Crucibulum crucibuliforme", "Lepista sordida", "Pholiota adiposa", "Xerocomellus cisalpinus", "Gymnopus peronatus", "Hypholoma marginatum", "Sparassis crispa", "Russula paludosa", "Suillus cavipes", "Mycena pelianthina", "Mycena polygramma", "Mutinus caninus", "Lactarius helvus", "Lycoperdon utriforme", "Russula xerampelina", "Lactarius tabidus", "Russula depallens", "Ganoderma adspersum", "Lycoperdon pratense", "Cortinarius anomalus", "Hygrophorus pustulatus", "Scleroderma verrucosum", "Hygrocybe reidii", "Entoloma serrulatum", "Russula rosea", "Mycena sanguinolenta", "Leucoagaricus leucothites", "Hygrocybe insipida", "Mycena pseudocorticola", "Leccinum versipelle", "Gliophorus irrigatus", "Clitocybe metachroa", "Cortinarius alboviolaceus", "Tricholoma fulvum", "Russula praetervisa", "Xerocomellus porosporus", "Lentinellus ursinus", "Lactarius pyrogalus", "Russula atropurpurea", "Pleurotus dryinus", "Russula solaris", "Tricholoma ustale", "Auriscalpium vulgare", "Russula queletii", "Hygrophorus eburneus", "Psilocybe coronilla", "Gymnopus perforans", "Lactarius pallidus", "Agrocybe pediades", "Amanita porphyria", "Leucopaxillus giganteus", "Leccinum variicolor", "Mycena metata", "Cuphophyllus flavipes", "Gymnopus aquosus", "Russula claroflava", "Russula violeipes", "Cyathus striatus", "Mycena arcangeliana", "Pluteus phlebophorus", "Mycena aetites", "Melanoleuca cognata", "Mycena olivaceomarginata", "Hygrocybe nitrata", "Ampulloclitocybe clavipes", "Lepiota magnispora", "Russula chloroides", "Russula fragilis", "Paxillus obscurisporus", "Pluteus romellii", "Scleroderma areolatum", "Coltricia perennis", "Cyanoboletus pulverulentus", "Hygrophorus olivaceoalbus", "Chrysomphalina grossula", "Clitopilus hobsonii", "Lactarius fluens", "Lepiota castanea", "Panaeolus acuminatus", "Russula velenovskyi", "Macrolepiota excoriata", "Hebeloma mesophaeum", "Pholiota lenta", "Paxillus filamentosus", "Lentinellus cochleatus", "Russula risigallina", "Cortinarius infractus", "Clitocybe phyllophila", "Coprinopsis nivea", "Mycena tenerrima", "Panaeolus semiovatus", "Russula foetens", "Lactarius aurantiacus", "Russula grata", "Hemipholiota populnea", "Leccinum cyaneobasileucum", "Russula nobilis", "Lactarius camphoratus", "Lactarius romagnesii", "Russula densifolia", "Russula cessans", "Laccaria bicolor", "Lycoperdon echinatum", "Russula aurora", "Inocybe sindonia", "Hericium cirrhatum", "Mycena abramsii", "Lactarius circellatus", "Conocybe aporos", "Flammulina elastica", "Lactarius vietus", "Omphalina pyxidata", "Mycena acicula", "Clavariadelphus pistillaris", "Cortinarius obtusus", "Amanita aureola", "Amanita eliae", "Amanita beckeri", "Amanita spissa", "Amanita ovoidea", "Amanita echinocephala", "Amanita aspera", "Amanita inaurata", "Amanita livido", "Amanita umbrinolutea", "Amanita crocea", "Volvariella surrecta", "Lepiota ventriosospora", "Lepiota alba", "Lepiota naucina", "Leucoagaricus holosericeus", "Leucocoprinus badhamii", "Cystoderma amianthinum", "Chamaemyces fracidus", "Limacella illinita", "Limacella guttata", "Catathelasma imperiale", "Coprinus truncorum", "Coprinus hemerobius", "Coprinus cinereus", "Agaricus macrosporus", "Agaricus bisporus", "Agaricus arvensis", "Agaricus abruptibulbus", "Agaricus perrarus", "Agaricus benesii", "Agaricus langei", "Agaricus vaporarius", "Agaricus radicatus", "Agaricus purpurascens", "Panaeolus sphinctrinus", "Panaeolus rickenii", "Anellaria semiovata", "Psathyrella candolleana", "Psathyrella prona", "Psathyrella spadiceo-grisea", "Psathyrella velutina", "Stropharia squamosa", "Stropharia eximia", "Agrocybe dura", "Agrocybe erebia", "Agrocybe paludosa", "Pholiota squarroso-adiposa", "Pholiota aurivella", "Pholiota destruens", "Pholiota flammans", "Pholiota carbonaria", "Pholiota flavida", "Pholiota amara", "Pholiotina blattaria", "Bolbitius vitellinus", "Gymnopilus spectabilis", "Hebeloma radicosum", "Hebeloma crustuliniforme", "Cortinarius traganus", "Cortinarius coerulescens", "Cortinarius nebularis", "Cortinarius zinziberatus", "Cortinarius collinitus", "Cortinarius violaceus", "Cortinarius arcuatorum", "Cortinarius glaucopus", "Cortinarius bulliardi", "Cortinarius hinnuleus", "Cortinarius pholideus", "Cortinarius nemorensis", "Dermocybe cinnamomea", "Dermocybe anthracina", "Dermocybe cinnabarina", "Inocybe maculata", "Inocybe grammata", "Inocybe umbratica", "Inocybe sambucina", "Inocybe godeyi", "Inocybe cincinnata", "Inocybe hirsuta", "Entoloma aprile", "Entoloma nitidum", "Entoloma mougeoti", "Entoloma mammosum", "Entoloma staurosporum", "Clitopilus cretatus", "Rhodocybe truncata", "Pluteus atricapillus", "Pluteus leoninus", "Collybia dryophila", "Collybia platyphylla", "Collybia exculpta", "Collybia acervata", "Collybia peronata", "Collybia hariolorum", "Oudemansiella longipes", "Strobilurus stephanocystis", "Strobilurus tenacellus", "Marasmius scorodonius", "Marasmius wynnei", "Marasmius bulliardii", "Marasmius androsaceus", "Marasmiellus ramealis", "Micromphale brassicolens", "Micromphale foetidum", "Mycena rosella", "Mycena callopoda", "Mycena flavipes", "Leucopaxillus amarus", "Tricholoma georgi", "Tricholoma viridilutescens", "Tricholoma flavovirens", "Tricholoma auratum", "Tricholoma albobrunneum", "Tricholoma batschii", "Tricholoma bufonium", "Tricholoma imbricatum", "Tricholoma vaccinum", "Tricholoma virgatum", "Tricholoma tigrinum", "Tricholoma squarrulosum", "Tricholoma atrosquamosum", "Tricholoma orirubens", "Tricholoma album", "Tricholoma acerbum", "Lepista irina", "Lepista caespitosa", "Lepista rickenii", "Melanoleuca grammopodia", "Melanoleuca subbrevipes", "Melanoleuca strictipes", "Melanoleuca excissa", "Clitocybe inversa", "Clitocybe gilva", "Clitocybe vermicularis", "Clitocybe maxima", "Clitocybe costata", "Clitocybe alexandri", "Clitocybe inornata", "Clitocybe cerussata", "Lyophyllum immundum", "Lyophyllum infumatum", "Lyophyllum semitale", "Lactarius porninsis", "Lactarius cilicioides", "Lactarius scrobiculatus", "Lactarius zonarius", "Lactarius luteolus", "Lactarius trivialis", "Lactarius contorversus", "Lactarius pergamenus", "Lactarius fuliginosus", "Lactarius picinus", "Lactarius uvidus", "Russula aeruginea", "Russula parazurea", "Russula amoena", "Russula integra", "Russula mustelina", "Russula vitellina", "Russula fellea", "Russula illota", "Russula laurocerasi", "Russula pseudodelica", "Russula viscida", "Russula amethystina", "Russula puellaris", "Russula versicolor", "Russula acrifolia", "Russula pectinatoides", "Russula amoenolens", "Russula sanguinea", "Russula badia", "Asterophora lycoperdoides", "Hygrophorus pudorinus", "Hygrophorus penarius", "Hygrophorus leporinus", "Hygrophorus queletii", "Hygrophorus chrysodon", "Hygrophorus co", "Hygrophorus gliocyclus", "Hygrophorus dichrous", "Hygrophorus agathosmus", "Hygrophorus lucorum", "Hygrophorus hypotheius", "Hygrophorus leucophaeus", "Hygrophorus erubescens", "Hygrophorus discoideus", "Hygrophorus aureus", "Hygrocybe konradii", "Hygrocybe obrussea", "Hygrocybe brevispora", "Hygrocybe citrina", "Hygrocybe quieta", "Hygrocybe intermedius", "Hygrocybe coccineus", "Hygrocybe nigrescens", "Hygrocybe laeta", "Hygrocybe reai", "Hygrocybe ovina", "Camarophyllus niveus", "Camarophyllus russocoriaceus", "Cantharellus friesii", "Cantharellus amethysteus", "Cantharellus subcibarius", "Gomphus clavatus", "Cantharellus cinerus", "Xeromphalina campanella", "Gomphidius maculatus", "Chroogomphus rutilus", "Chroogomphus helveticus", "Pleurotus columbinus", "Pleurotus sapidus", "Pleurocybella porrigens", "Panus tigrinus", "Crepidotus variabilis", "Phylloporus rhodoxanthus", "Boletus fechtneri", "Boletus impolitus", "Boletus fragrans", "Boletus junquilleus", "Boletus albidus", "Boletus satanoides", "Boletus rhodoxanthus", "Boletus purpureus", "Boletus dupainii", "Xerocomus spadiceus", "Xerocomus parasticus", "Leccinum testaceo-scabrum", "Leccinum criseum", "Leccinum holopus", "Leccinum duriusculum", "Suillus aeruginascens", "Suillus placidus", "Gyrodon lividus", "Polyporus confluens", "Polyporus ovinus", "Polyporus cristatus", "Polyporus pes-caprae", "Polyporus giganteus", "Polyporus montanus", "Polyporus frandosus", "Polyporus intybaceus", "Polyporus squamosus", "Polyporus lentus", "Polyporus sulphureus", "Polyporus lepideus", "Polyporus mori", "Polyporus arcularius", "Polyporus badius", "Merulius tremellosus", "Inonotus dryadeus", "Osmoporus odoratus", "Daedalea quercina", "Daedalea confragosa", "Thelephora palmata", "Sarcodon badium", "Hydnum rufescens", "Calodon aurantiacum", "Calodon compactum", "Dryodon erinaceus", "Pleurodon auriscalpium", "Sparassis laminosa", "Clavaria flava", "Clavaria aurea", "Clavaria rufescens", "Clavaria formosa", "Clavaria pallida", "Clavaria lavendula", "Clavaria cristata", "Clavaria rugosa", "Clavaria fennica", "Clavaria invalii", "Clavaria stricta", "Clavaria vermicularis", "Pterula multifida", "Calocera viscosa", "Clavaria pistillaris", "Clavaria truncata", "Tremellodon gelatinosum", "Tremella mesenterica", "Exidia galndulosa", "Tremiscus helvelloides", "Lycoperdon caelatum", "Lycoperdon mammaeforme", "Lycoperdon saccatum", "Lycoperdon umbrinum", "Bovista plumbea", "Rhizopogon roseolus", "Scleroderma geaster", "Geastrum triplex", "Geastrum fimbriatum", "Geastrum pectinatum", "Astraeus hygrometricus", "Clathrus cancellatus", "Phallus hadriani", "Sebacina laciniata", "Spathularia flavida", "Bulgaria inquinans", "Lycogala epidendron", "Arcyria denudata", "Tubifera ferruginosa", "Fuligo septica", "Xylaria polymorpha", "Xylaria hypoxylon", "Morchella deliciosa", "Morchella pragensis", "Mitrophora hybrida", "Verpa bohemica", "Verpa digitaliformis", "Helvella crispa", "Helvella lacunosa", "Helvella elastica", "Gyromitra fastigiata", "Tuber brumale", "Tuber melanosporum", "Elaphomyces granulatus", "Choiromyces maeandriformis", "Otidea onotica", "Caloscypha fulgens", "Aleuria aurantia", "Sarcoscypha coccinea", "Scutellinia scutellata", "Paxina acetabulum", "Peziza vesiculosa", "Peziza badia", "Discina perlata", "Disciotis venosa", "Sclerotinia tuberosa", "Agaricus fuscofibrillosus", "Aleuria boudieri", "Aleuria flavorubens", "Aleurocystidiellum subcruentatum", "Amanita friabilis", "Amanita lepiotoides", "Amanita pachyvolvata", "Amanita solitaria", "Amanita vittadinii", "Amylocorticium subincarnatum", "Amylocystis lapponicus", "Anomoporia bombycina", "Antrodia crassa", "Antrodia variiformis", "Antrodiella citrinella", "Antrodiella fragrans", "Arrhenia spathulata", "Ascobolus behnitziensis", "Ascobolus epimyces", "Ascotremella faginea", "Atheloderma mirabile", "Auriporia aurulenta", "Baeospora myriadophylla", "Bisporella calycellinoides", "Bolbitius lacteus", "Boletus adonis", "Boletus depilatus", "Boletus torosus", "Callistosporium donadinii", "Callistosporium elaeodes", "Callistosporium luteoolivaceum", "Calocybe carnea", "Calocybe constricta", "Calocybe ionides", "Calocybe obscurissima", "Camarophyllopsis foetens", "Camarophyllopsis micacea", "Camarophyllopsis phaeophylla", "Campanella caesia", "Ceriporiopsis jelicii", "Ceriporiopsis myceliosa", "Ceriporiopsis subrufa", "Chalciporus rubinus", "Cheilymenia magnifica", "Ciboria aestivalis", "Clavulicium macounii", "Clitocybe collina", "Clitocybula lacerata", "Contumyces rosellus", "Coprinus strossmayeri", "Cordyceps bifusispora", "Cordyceps intermedia", "Cordyceps longisegmentis", "Cordyceps riverae", "Cordyceps sphecocephala", "Cortinarius bulliardii", "Cortinarius ionochlorus", "Cotylidia pannosa", "Crinipellis subtomentosa", "Cudonia circinans", "Cystoderma superbum", "Dacryobolus karstenii", "Dentipellis fragilis", "Dermoloma cuneifolium", "Dermoloma josserandii", "Dermoloma pseudocuneifolium", "Desmazierella acicola", "Discina fastigiata", "Discina leucoxantha", "Discina montana", "Discina parma", "Disciseda bovista", "Disciseda candida", "Elaphomyces anthracinus", "Endoptychum agaricoides", "Entoloma bloxamii", "Entoloma caesiocinctum", "Entoloma catalaunicum", "Entoloma corvinum", "Entoloma cyanulum", "Entoloma kervernii", "Entoloma opacum", "Entoloma plebeioides", "Entoloma plebejum", "Entoloma porphyrophaeum", "Entoloma pseudoturci", "Entoloma reinwaldii", "Entoloma rhombisporum", "Entoloma saundersii", "Faerberia carbonaria", "Flammulina ononidis", "Flavophlebia sulphureoisabellina", "Fomitopsis rosea", "Fomitopsis spraguei", "Galerina jaapii", "Galerina paludosa", "Galerina tibiicystis", "Gastrosporium simplex", "Geastrum minimum", "Geoglossum cookeanum", "Geoglossum glutinosum", "Geoglossum umbratile", "Geopora nicaeensis", "Geopyxis majalis", "Gerhardtia piperata", "Gloeocystidiellum subasperisporum", "Gomphidius roseus", "Gymnopilus bellulus", "Gymnopilus picreus", "Haasiella splendidissima", "Haasiella venustissima", "Hapalopilus croceus", "Hapalopilus salmonicolor", "Hebeloma gigaspermum", "Helvella albella", "Helvella branzeziana", "Helvella cupuliformis", "Helvella helvellula", "Helvella lactea", "Helvella phlebophora", "Helvella queletiana", "Helvella spadicea", "Hericium flagellum", "Hexagonia nitida", "Heyderia abietis", "Holwaya mucida", "Humaria aurantia", "Hydropus atramentosus", "Hygrocybe calciphila", "Hygrocybe calyptriformis", "Hygrocybe citrinovirens", "Hygrocybe coccineocrenata", "Hygrocybe colemanniana", "Hygrocybe fornicata", "Hygrocybe ingrata", "Hygrocybe intermedia", "Hygrocybe irrigata", "Hygrocybe lacmus", "Hygrocybe perplexa", "Hygrocybe russocoriacea", "Hygrocybe sciophanoides", "Hygrocybe spadicea", "Hygrocybe splendidissima", "Hygrocybe subpapillata", "Hygrophorus calophyllus", "Hygrophorus camarophyllus", "Hygrophorus capreolarius", "Hygrophorus hyacinthinus", "Hygrophorus hypothejus", "Hygrophorus mesotephrus", "Hyphoderma cremeoalbum", "Hyphoderma guttuliferum", "Hyphoderma litschaueri", "Hyphoderma macedonicum", "Hypholoma elongatum", "Hypholoma myosotis", "Inonotus dryophilus", "Inonotus nidus-pici", "Jaapia ochroleuca", "Junghuhnia collabens", "Junghuhnia fimbriatella", "Kuehneromyces lignicola", "Laccaria pumila", "Lactarius acris", "Lactarius cistophilus", "Lactarius lacunarum", "Lactarius lilacinus", "Lactarius omphaliformis", "Lactarius tesquorum", "Lambertella corni-maris", "Lentinus degener", "Leptosporomyces roseus", "Leucoagaricus pilatianus", "Leucogaster nudus", "Leucopaxillus compactus", "Leucopaxillus macrocephalus", "Lyophyllum rhopalopodium", "Lyophyllum transforme", "Marasmiellus humillimus", "Marasmius anomalus", "Marasmius buxi", "Marasmius caricis", "Marasmius collinus", "Marasmius cornelii", "Marasmius hudsonii", "Marasmius menieri", "Marasmius undatus", "Melanogaster vittadinii", "Metulodontia nivea", "Microglossum olivaceum", "Microglossum viride", "Mitrula paludosa", "Moellerodiscus iodotingens", "Moellerodiscus tenuistipes", "Mollisia olivascens", "Mucronella bresadolae", "Mucronella calva", "Mycena latifolia", "Mycenastrum corium", "Myriosclerotinia dennisii", "Myriostoma coliforme", "Oligoporus cerifluus", "Oligoporus lowei", "Oligoporus placenta", "Ombrophila rivulorum", "Omphalina baeospora", "Omphalina chrysophylla", "Omphalina obatra", "Onygena equina", "Orbilia polyspora", "Ossicaulis lignatilis", "Otidea propinquata", "Pachyella peltata", "Pachyella pseudosuccosa", "Pachyella punctispora", "Pachyella violaceonigra", "Parmastomyces mollissimus", "Peziza alborosea", "Peziza ammophila", "Peziza apiculata", "Peziza boltonii", "Peziza buxea", "Peziza flavida", "Peziza merdae", "Peziza muscicola", "Peziza nivalis", "Peziza obtusapiculata", "Peziza saccardiana", "Peziza subuliginosa", "Peziza subumbrina", "Phaeocollybia jennyi", "Phaeomarasmius limulatus", "Phaeomarasmius muricatus", "Phallogaster saccatus", "Phellinus nigrolimitatus", "Phellinus pouzarii", "Phlebia albida", "Phlebia centrifuga", "Phlebia cornea", "Phlebia griseoflavescens", "Phlebia subulata", "Phylloporus pelletieri", "Phyllotopsis nidulans", "Piptoporus quercinus", "Piptoporus soloniensis", "Pisolithus arhizus", "Plectania melastoma", "Plectania platensis", "Pleurotus calyptratus", "Pleurotus cornucopiae", "Poronia punctata", "Porphyrellus porphyrosporus", "Psathyrella ammophila", "Psathyrella melanthina", "Psathyrella typhae", "Pseudoomphalina kalchbrenneri", "Pseudoplectania melaena", "Pseudorhizina sphaerospora", "Psilopezia nummularialis", "Ptychoverpa bohemica", "Pulveroboletus hemichrysus", "Pulveroboletus lignicola", "Pulvinula globifera", "Pulvinula laeterubra", "Pulvinula ovalispora", "Pycnopeziza sejournei", "Pycnoporellus fulgens", "Pyronema dubium", "Resupinatus vetlinianus", "Rhodocybe fallax", "Rhodoscypha ovilla", "Rigidoporus crocatus", "Rodwayella sessilis", "Russula alnetorum", "Sarcoscypha macaronesica", "Scleroderma polyrhizum", "Scutellinia pilatii", "Scutellinia tuberculata", "Scytinostroma galactinum", "Scytinostroma odoratum", "Scytinostromella heterogenea", "Skeletocutis odora", "Skeletocutis papyracea", "Skeletocutis stellae", "Sparassis spathulata", "Spongipellis delectans", "Steccherinum subcrinale", "Stropharia dorsipora", "Strossmayeria rackii", "Suillus lakei", "Suillus tridentinus", "Tatraea dumbirensis", "Trichoglossum confusum", "Trichoglossum hirsutum", "Trichoglossum variabile", "Tricholoma aurantium", "Tricholoma caligatum", "Tricholoma colossus", "Tricholoma goniospermum", "Tricholoma luridum", "Tulostoma fimbriatum", "Tulostoma melanocyclum", "Urnula craterium", "Veluticeps ambigua", "Verpa conica", "Vibrissea truncorum", "Xerocomus moravicus", "Xerula caussei", "Psilocybe semilanceata", "Psilocybe serbica", "Psilocybe cyanescens", "Panaeolus cinctulus"]
print(len(VRSTE), 'vrsta u atlasu')

In [ ]:
# Testne fotografije iz atlasa: model ih ne smije vidjeti dok uči (inače test ne vrijedi)
TEST_DATOTEKE = set(["File:2005-03-28 Strobilurus esculentus.jpg", "File:2007-08-09 Lycoperdon pratense.jpg", "File:2007-08-29 Clitocybe rivulosa.jpg", "File:2007-10-20 Cuphophyllus virgineus.jpg", "File:2008-07-07 Amanita excelsa colors corrected.jpg", "File:2008-10-11 Lepista sordida 20505 cropped.jpg", "File:2008-10-19 Hygrocybe nitrata 20200 cropped.jpg", "File:2009-06-06 Hygrocybe laeta 2 21847.jpg", "File:2009-09-15 Cortinarius elatior Fr 57078 crop.jpg", "File:2009-09-25 Hericium coralloides (Scop.) Pers 58068 crop.jpg", "File:2009-09-29 Xerocomellus pruinatus.jpg", "File:2009-10-23 Leucoagaricus leucothites (Vittad.) M.M. Moser ex Bon 61894 crop.jpg", "File:2010-04-27 Inocybe erubescens.jpg", "File:2010-08-11 Hygrophorus olivaceoalbus 2.jpg", "File:2010-08-11 Russula claroflava Grove. ss. Melz, & Zv., J. Schff 98988.jpg", "File:2010-09-06 Russula laurocerasi Melzer 122171.jpg", "File:2010-10-08 Ampulloclitocybe clavipes cropped.jpg", "File:2010-11-06 Pholiota gummosa (Lasch) Singer 118899.jpg", "File:2010-11-20 Cuphophyllus pratensis.jpg", "File:2010-12-22 Russula cessans A. Pearson 127746.jpg", "File:2011-05-14 Russula lepida 146790 146793 combined.jpg", "File:2011-06-17 Hygrocybe cantharellus 69428 cropped.jpg", "File:2011-10-05 Lentinellus ursinus (Fr.) Kühner 172682.jpg", "File:2011-11-08 Russula xerampelina (Schaeff.) Fr 180998 mod.jpg", "File:2011-11-12 Pluteus salicinus (Pers.) P. Kumm 181966.jpg", "File:2011-12-01 Lepiota cristata (Bolton) P. Kumm 186612.jpg", "File:2012-01-19 Ganoderma pfeifferi 196230 (cropped).jpg", "File:2012-05-12 Russula velutipes Velen 218452.jpg", "File:2012-06-15 Russula risigallina (Batsch) Saccardo 228083.jpg", "File:2012-06-15 Russula velenovskyi Melzer & Zvára 228092.jpg", "File:2012-08-24 Boletus fuscoroseus Smotlacha 253660 crop.jpg", "File:2012-08-25 Lactarius circellatus Fries 254180 crop.jpg", "File:2012-10-22 Cyathus striatus (Huds.) Willd 274333.jpg", "File:2012-10-26 Mycena pelianthina (Fr.) Quél 320254.jpg", "File:2012-10-28 Cortinarius croceus group 276418 (cropped).jpg", "File:2012-12-14 Laccaria proxima (Boud.) Pat 293250.jpg", "File:2014-02-18 Coprinopsis nivea (Pers.) Redhead, Vilgalys & Moncalvo 404556.jpg", "File:2026-07-24 Polyporus tuberaster (Z5-10330E) by Achim-Lammerts.jpg", "File:AD2009Sep06 Ramaria flava 01.jpg", "File:Agaricus impudicus 01.jpg", "File:Agaricus silvicola father and son.jpg", "File:Agaricus.bitorquis.nathan.jpg", "File:Agrocybe pediades Baja California.jpg", "File:Albatrellus cristatus BS11.jpg", "File:Amanita fulva lowpx.jpg", "File:Amanita gemmata von hms.jpg", "File:Amanita porphyria Alb. & Schwein 355243 2013-08-06 (cropped).jpg", "File:AmanitaVaginata2.jpg", "File:Armillaria gallica 26659.jpg", "File:Armillaria ostoyae.jpg", "File:Auriscalpium vulgare 129877.jpg", "File:Blaufüßiger Raufuß Leccinum cyaneobasileucum.jpg", "File:Bol.poro.jpg", "File:Boletus aereus Italy.jpg", "File:Boletus cavipes.jpg", "File:Boletus edulis IT.jpg", "File:Boletus pinophilus3.JPG", "File:Boletus pulverulentus 2011 G1.jpg", "File:Boletus radicans-20.7.2012-1.JPG", "File:Boletus.JPG", "File:Calocybe gambosa 080420wa.jpg", "File:Chlorophyllum rhacodes LC0093.jpg", "File:Clathrus ruber spacepleb.jpg", "File:Clavariadelphus-pistillaris.jpg", "File:Clavulina cristata sasata.JPG", "File:Clavulina rugosa LC0369.jpg", "File:Clitocybe nuda (Fr.) H.E. Bigelow & A.H. Sm 267650.jpg", "File:Clitocybe phyllophila T71 (1).jpg", "File:Clitocybe-fragrans-duft-trichterling.jpg", "File:Clitocybe-odora.jpg", "File:Collybia dryophila 20061001w.jpg", "File:Collybia peronata 20070812wb.JPG", "File:Coloured Figures of English Fungi or Mushrooms - t. 134.jpg", "File:Coprinellus disseminatus group.jpg", "File:Coprinellus micaceus Glimmer-Tintling.jpg", "File:Coprinopsis atramentaria 3 - Lindsey.jpg", "File:Coprinus comatus, the shaggy ink cap, lawyer's wig, or shaggy mane mushroom.jpg", "File:Corellanus.jpg", "File:Cortinarius anomalus - Lindsey.jpg", "File:Cortinarius flexipes LC0111.jpg", "File:Cortinarius infractus 67718.jpg", "File:Cortinarius praestans.JPG", "File:Cortinarius-alboviolaceus-0203.jpg", "File:Crimson Waxcap. Hygrocybe punicea (49055268862).jpg", "File:Cuphophyllus flavipes - Yellow Foot Waxcap (48812855077).jpg", "File:Dehnbare Helmling Mycena epipterygia.jpg", "File:Entoloma serrulatum (Fr.) Hes 772415.jpg", "File:Entoloma sinuatum group.JPG", "File:Fichten-Reizker Lactarius deterrimus.jpg", "File:Fistulina hepatica.JPG", "File:Fleischblasse Milchling Lactarius pallidus.jpg", "File:Fomes fomentarius 2010 G2.jpg", "File:Fomitopsis pinicola 109142.jpg", "File:Galerina marginata Point Reyes.jpg", "File:Ganoderma applanatum 2010 G1.jpg", "File:Ganoderma australe.jpg", "File:Geastrum triplex - Bola Creek.jpg", "File:Gemeine Hundsrute Mutinus caninus.JPG", "File:Gomba.jpg", "File:Grote sponszwam (Sparassis crispa). 09-11-2023. (d.j.b).jpg", "File:Gymnopilus sapineus group 568289.jpg", "File:Gymnopus.confluens.-.lindsey.jpg", "File:Habichtspilz-Sarcodon-imbricatus.jpg", "File:Haematopus-on-oak.jpg", "File:Handkea utriformis G25 lights-shadows.jpg", "File:Hasenpfoten.jpg", "File:Hebeloma mesophaeum - Lindsey.jpg", "File:Hedgehog fungi2.jpg", "File:Hemipholiota populnea 26704609.jpg", "File:Hericium cirrhatum. The Tiered Tooth. Kilmaurs, East Ayrshire.jpg", "File:Honig-Saftling Hygrocybe reidii.JPG", "File:Hortiboletus bubalinus 565239.jpg", "File:Hygrocybe chlorophana.jpg", "File:Hygrocybe coccinea (1).jpg", "File:Hygrocybe conica (2005 11 07) 1.jpg", "File:Hygrocybe irrigata.jpg", "File:Hygrocybe miniata - Ferndale Park.jpg", "File:Hygrocybe psittacina 54530.jpg", "File:Hygrophorus eburneus-pastorino.JPG", "File:Hygrophorus nemoreus 20061029wb.jpg", "File:Hygrophorus poetarum, Forchtenstein, top crop.jpg", "File:Hypholoma-capnoides.jpg", "File:Ino.geo.jpg", "File:Laccaria amethystina - Lakówka ametystowa.jpg", "File:Laccaria, DOE.jpg", "File:Lacpub2big.jpg", "File:Lacrymaria lacrymabunda (Bull.) Pat 173183 crop.jpg", "File:Lact.chry.jpg", "File:Lact.ruf.jpg", "File:Lact.sub.jpg", "File:Lactarius blennius JPG01.jpg", "File:Lactarius camphoratus - Lindsey.jpg", "File:Lactarius fluens - Lindsey.jpg", "File:Lactarius helvus - Lindsey.jpg", "File:Lactarius hepaticus 031123w.jpg", "File:Lactarius piperatus 98569.jpg", "File:Lactarius pyrogalus.jpg", "File:Lactarius quietus 2010 G1.jpg", "File:Lactarius romagnesii 41561-cropped.jpg", "File:Lactarius semisanguifluus1597.jpg", "File:Lactarius torminosus (cropped).jpg", "File:Lactarius turpis 2010 G1.jpg", "File:Lactarius041031w.jpg", "File:Laetiporus sulphureus big.jpg", "File:Leccinum pseudoscabrum 2020 G1.jpg", "File:Leccinum variicolor LC0365.jpg", "File:Leccinum versipelle LC0366.jpg", "File:Lepiota castanea 70848.jpg", "File:Lepiota magnispora 32236348.jpg", "File:Lepista personata.JPG", "File:Leucoagaricus nympharum1.jpg", "File:Leucopaxillus giganteus 20070817wa.JPG", "File:Lycoperdon echinatum 54507 crop.jpg", "File:Lycoperdon nigrescens - Lindsey.jpg", "File:Marasmius alliaceus 20070928wb.JPG", "File:Marasmius rotula 148433.jpg", "File:Melanoleuca cognata Frühlings Weichritterling.jpg", "File:Micromphale.perforans.-.lindsey.jpg", "File:Milde Milchling (Lactarius aurantiacus).JPG", "File:Mycena 60303.jpg", "File:Mycena acicula Oakland.jpg", "File:Mycena adscendens 9244.jpg", "File:Mycena aetites.png", "File:Mycena arcangeliana 107930.jpg", "File:Mycena crocata.jpg", "File:Mycena epipterygia - Lindsey 7a.jpg", "File:Mycena galopus 68060.jpg", "File:Mycena metata.png", "File:Mycena rosea, Rosy Bonnet, UKB.jpg", "File:Mycena sanguinolenta Van Damme.jpg", "File:Mycena spec. - Lindsey 16a.jpg", "File:Mycena tintinabulum.jpg", "File:Nabeling Geroell002.JPG", "File:Panaeolus acuminatus.jpg", "File:Panaeolus foenisecii Cupertino.jpg", "File:Panaeolus papilionaceus California.jpg", "File:Panaeolus semiovatus LC0334.jpg", "File:Paxillus filamentosus 25465.jpg", "File:Paxillus obscurisporus (35978017564).jpg", "File:Phaeolepiota aurea 10866-83fabfaacdf92ca2ba5a395f175754a4.jpg", "File:Phaeolus schweinitzii 433858897.jpg", "File:Phallus impudicus LC0235.jpg", "File:Pholiota aurivella 56647.jpg", "File:Pleurotus dryinus JPG1.jpg", "File:Pleurotus pulmonarius LC0228.jpg", "File:Plooivoetstuifzwam (Lycoperdon excipuliforme). Locatie, Hortus (Haren, Groningen) 10.jpg", "File:Post-1-1127312702.jpg", "File:Pseudoclitcybe cyathiformis.jpg", "File:Pseudocraerellus sinuosus1.jpg", "File:Pseudohydnum gelatinosum Abernethy Forest.jpg", "File:Ramaria stricta 171867.jpg", "File:Rehbrauner Dachpilz Bruderwald-20251017-RM-164830.jpg", "File:Rettichhelmling002.JPG", "File:Rhodocollybia.maculata.-.lindsey.jpg", "File:Roodbruine schijnridderzwammen (Lepista flaccida). 22-01-2021 (d.j.b.) 01.jpg", "File:Russ nobilis.jpg", "File:Russ.atro.jpg", "File:Russula chloroides.jpg", "File:Russula densifolia gołąbek gęstoblaszkowy BS11.JPG", "File:Russula foetens.JPG", "File:Russula fragilis.jpg", "File:Russula nigricans - Lindsey 3a.jpg", "File:Russula ochroleuca châtaigne.JPG", "File:Russula solaris.JPG", "File:Russula violeipes 243363.jpg", "File:Schizophyllum commune (Split gill) (33389628036).jpg", "File:Schwarzpunktierte Schneckling Hygrophorus pustulatus.jpg", "File:Scleroderma areolatum.jpg", "File:Scleroderma verrucosum 01.JPG", "File:Scutiger pes-caprae 31566.jpg", "File:Sparrige-Schüppling (Pholiota squarrosa).JPG", "File:Stachelbeer-Täubling Russula queletii.JPG", "File:Stereum hirsutum Aberystwyth 2024.jpg", "File:Stockschwaemmchen.jpg", "File:Stropharia caerulea a2 (2).JPG", "File:Stropharia coronilla 20070819wa.JPG", "File:Stropharia semiglobata 38287.jpg", "File:Thelephora.terrestris.-.lindsey.jpg", "File:Tiny Russula paludosa (Appelrussula) at Rozendaal castle park - panoramio.jpg", "File:Tonweißer Schüppling Pholiota lenta.jpg", "File:Trametes gibbosa JPG1.jpg", "File:Trametes hirsuta BS14.2.jpg", "File:Trametes ochracea a3 (8).jpg", "File:Tricholoma argyraceum 56.jpg", "File:Tricholoma fulvum LC0373.jpg", "File:Tricholoma scalpuratum 20061014wa.jpg", "File:Tricholoma sulphureum 031123w.jpg", "File:Tricholoma terreum 20061105wa.jpg", "File:Tricholoma.ustale.-.lindsey.jpg", "File:Volvopluteus SoCal.jpg", "File:Xerula.radicata3.-.lindsey.jpg", "File:Zitzen-Riesenschirmling Macrolepiota mastoidea.JPG"])
TEST_HASH = [int(h, 16) for h in ["bc9783531517531a", "93cc6c4c8c2c71db", "b2ee66969a8ec6cb", "79f9fc7632b090d1", "6b4b296b216b4acc", "4d8e263134654dcf", "876731319b3939de", "f1e96b4916009958", "2ff0488c9292d444", "edf2d2b657979930", "f237a731692913d1", "0b4e4da9ab22b7d5", "16353d2dba333599", "ce8c5d4d4d27a29c", "bb99092b276353b3", "33c5c553298dab33", "539733e362498d8c", "de70e0cce8ec1c13", "2d53acb983656372", "34661687b3b1b9c1", "624e9ebd46da9f2a", "355b93337b2d8149", "0f2c7e0d35558d32", "a7990e3c91b2b2dc", "9d89696a2c2c4ec9", "ed472529890a4332", "713342d9890d4c4e", "ad333133a7a55796", "939c0d291b676364", "034d1c2eb7942697", "d02d26666744cc81", "3363a3a77597534a", "599f3f6327351b8c", "06939aaba12d21b1", "d3e62333332a242a", "cf0b3459b1a9cd46", "3bec4c6dc933332a", "276e6635512d6be3", "64993364335a7869", "b6e4918d6c2c2b54", "529323323451caef", "9d2671552d69d186", "19797d13838d58ee", "f9b0b9baab69d9b9", "8d367bdbd7c6a5c3", "cd4e6b45c19b1a2a", "554e6c292d696962", "27353963111ba695", "596b9491919d4919", "9f37733333565b73", "254d8b39254c7777", "365bc966665b7362", "331b33b2765551d4", "1c17076c2c1adae6", "f4e6e6d3b3d86233", "5bd69feebd0fe490", "19339990a2cbcb49", "9aa316f56557b3f2", "8d8b3931ce8e6282", "6d649a1936329ace", "e24ccc47cdcd2eba", "362e4eac35356d27", "2c4c073b3fe7b3cc", "46073b1b074cf8e7", "92871319353d1cb0", "da4b4319252129d1", "24662611155d33b7", "c9843735a9ab2334", "3939393625292274", "57b576cc4d779484", "921a793b939b3338", "5e4b63595157c6f4", "8fc6c6e7373a6838", "4b2d3a3bb3746484", "8e060531236b4a59", "1466771b0d91b5b4", "0d4c5b2929495a38", "8a543763959966d6", "9d1d4d495616b5cd", "56466d2d1bada6ae", "751423795c371d1b", "d1a4a61c1ea51f1e", "8f8d096d27667656", "6b4797b30f45654b", "ac1b5d2f3a2e2f64", "36a6468bbb3327ad", "da142666e5c56c28", "d4d74b533193696f", "60309e07250fcfe5", "6ede9ebddbcc4967", "1b13a38b8f8db2b8", "f49b35234a3254dc", "b81e263b8b1b9ec8", "8e5e06232d656517", "5768cce4351d8ec7", "45cc3d6d31859bca", "030c8a33230b3935", "93b3262f266e4c86", "890d8f2f272bc9ce", "d9934b4b1773a8d4", "d81b4363731b39a5", "7979991d1ccaab39", "bd3c46b9bb2a3409", "e48a2b37296ddccc", "d86f0fcd2f337d8e", "3f4f8f39131c666f", "438f3b9191456969", "ebf66d71e7f17848", "139423272f231346", "ac0e7a3912723a39", "9316968b4b1f3837", "8ca68bcd8b9c3560", "c6776bb33d5f6d4a", "33ec9b2b1b37a40b", "397577333637be1d", "1443ab3637df5857", "1207278fef4cccce", "0bcf1b5b71ad9f95", "5b9171b96b1bb693", "2e6c656169298994", "27866765612d2549", "048f2fe7a7256541", "375a13d3d99356e4", "e6c617375637a5ba", "9b337333b666458d", "8173cb2d331b309a", "5d6d0f2e2b2b6b9a", "961e7b69492b26cd", "67db1b5a9a8e366a", "b3333333232b63d3", "a5375aa5e9e3c2c8", "cbc1d4f98b826348", "bb4a682e6537b3b9", "1a5db9ca31b3464c", "a6a7263637a7c19d", "2a2d390d276f0b19", "f22e3961571bae16", "a33edb131b33e5b5", "d90f0e33c34ee4b5", "6512312d2d0d8ceb", "cc19372b5333cb4c", "d6674bc5edb9b445", "393341c929293354", "ec2d3b824f8686b0", "036546763e8ed9f2", "4b0f564226948d28", "37bccc173c183939", "9b9a5165474b90b4", "d779a9ba9ac68d98", "2e0d0cdf9e2b339a", "161637230b27c7c2", "b7c697333237644c", "6f3956a7a1f2e46d", "4c8f133f1e3e1c73", "6584973327070d70", "6362d8d8d881c34b", "c1c3b1d99b225aee", "eca8986d6666defa", "2c4c073b3fe7b3cc", "9b1313265b1ea6b5", "9b9a18b95cccccb6", "7363d34ec9491a92", "8e36060d6e2c3993", "b5930d4b0b3b0b9e", "e6191cc335775952", "4c363d86a7a52c28", "ab333338cd652575", "9183939a9e16db74", "4a3b1b2363138bfa", "cb4a4ed695e6c55d", "1b33260e995b9d86", "ddda6c2dc9195152", "d5963775530a8c43", "a8c0656464a1c02e", "519d2f1736164683", "1b3c79769839331a", "73079f636e6616e6", "652302a98fbb4a58", "59b0dcf4e4d4d0f0", "52929b1d71735414", "e64b8d2f4f0c6ee5", "2d6699332d0dcece", "a36965132c257a9a", "4c1f07241c0d91de", "2c0bcd0c164d89c4", "dd0d1a93119b8beb", "c60f0dcdad4c4467", "63cd9fcfb3b3395d", "1a18461332737368", "140c4e4624060621", "8c4c43564f4db60e", "272b641e4e0e0ea6", "9637365a1b9bcf4c", "e24ccc47cdcd2eba", "32332a76772d2ec6", "968727b63367d61f", "2c87c3cde9f0c425", "e5f61e976b07352a", "4ce9754e6959cc2e", "191a3731787e35a6", "95a6862f139a4e2b", "5976864b0989c0e9", "daeee7679e1a98da", "25313d2d2d66634b", "450661f16358d80c", "3c1e968f66e2e1e3", "6763733266921d46", "38353355f5a7a3d3", "c3739697b2418ece", "8d1713c332326c4b", "92b973d28e573a8e", "6f1d394b93d9ed27", "6d0b312d3b333522", "34141e9e8da3b069", "a33a0c26939606a7", "4a4b9686c94ed2a8", "c64e553a8b19c942", "b632228e8aaaa625", "ad36eecbcccd999d", "ddf6b6f626a62636", "929b0f4d231e5ae4", "97a58d9d1d65cc91", "706a6f878ec5d5e2", "2040c82694aec660", "33613e26860d3030", "094b692081e06c19", "bcdb9b99935b9a9a", "6d180d1733694144", "8d256d2d2ea49cce", "9fa32de4a3616bce", "96570f0e9e1eddc1", "9bb3337171553b37", "d45a59352d9cbc6c", "915ee3a325331c87", "d70a2d2c2d934c95", "1c2c870333358f8d", "474f1333733988ca", "323c0d0b9b83c38c", "909296cc0d85ddd9", "a5a595985a525653", "a6267b9743640c99", "cb2d33130f161794", "12091f1cc8d26ccd", "43554c222cc2c29c", "68568633f0bbb2b2", "5d4d6d79b9140c47", "1ac49bbab39f828c", "a7667c2da5b1909c", "b6da8b4b6963a68d", "952531212b260465", "98286852352bc7cd", "42471e5a4e4ec577", "1f2caf1f4f8967cd", "d09c1f393d3597b3", "9b1e9959598999c1", "e7c5b5253185a5c5", "407614951f1e5b67", "e3cc4c898b8adc10", "3337370f2f793b0e", "2d9a1c6c29322735", "4fcfb3a6a6a76258", "47095a73b325095b", "33591c53979d8e97", "d673278d4dc9e376", "cc8f9b121e3834dc", "238e1a333529a312", "db9333d347657177", "304b2b93b1b51b58", "99d3a7375777438e", "524ca909914a2e0e", "24321b0b13838919", "9923034b49993836", "ca5bb098c8584868", "130f93091fd77637", "870103871727b725", "3613193514955b59", "551a03d3d3946cd9", "b9c586070d0d6c57", "6d2ccf1e6cea6a70", "94878b2627073611", "1a26062627028236", "0e1e4e4d6f8e8dca", "82991c1643084263", "19373e3293933335", "0d06cb0b1b4b4970", "8c1f3b3f23270559", "0d2e3e93d32b9b1b", "0dc961f363616963", "630f237d33879792", "888c8e0e0e0c4d32", "c6ca0c2f25676717", "7079d37737135a24", "4c45072208e0e1f8", "2c2e45661b88e666", "ede27373b1b12626", "afecbd140fa72607", "f3b9e9a4b8c8e6b4", "008d8f0b6b654345", "666478336347c450", "120fcd35a7265a4b", "b6a3018f8cdd8d0f", "3b1f260e9d91b5f6", "9a923833c39efbed", "6fb59d8ccc52183a", "972f336b72a295e1", "dcf4371fe9494f93", "6223b3333bd39b17", "7c2d6414356d5832", "4f072d3d148c8e8c", "34d38dc7b22e6b58", "cfcf020a83cc2186", "3011094d4c120646", "9d3d333136dbe24e", "9d3c3323a48443c6", "d6c6f033cffa922b", "93276472a429b989", "d39c1cd9d197ad34", "133bfd4b56964be3", "13959d2f0621439e", "29191a39292928d4", "3d323f6f5a334101", "62638553527c7ad8", "9d3c12c396961623", "1c8e2a078f5a466c", "b29b2367a2a6ccda", "122d4969339bc743", "34366f161713c36a", "c1cc9c3765195ac9", "8e0f178c4dcd8f29", "cba993b1ef896161", "cf67bbb9de5a4f7b", "292a6475316569a9", "7f9d971f3e3b7c6e", "9ac40f6b6470c7f9", "304c0e3913337377", "2b379f5a1b8da426", "5370b8d494ddcd4b", "c34b3d8f8f6d79f3", "21951d3b1bf37379", "480f16130b074dc2", "26060f0b139364c9", "3b5f771103c8b8cd", "0d2cca95919a1bd3", "ca9d1f1b9b9a2626", "cb6b33132d8cd9fd", "9bc7ce49278e6e96", "6545062f632b2b87", "b696a63351513336", "eb652d279b9c0c38", "86864d2c6d216a4d", "cc0b692d0b4a2ca9", "a5161ea3b3c6d484", "561c0d0d4f8d8d29", "aece0f2b86cecec8", "2b67537172763662", "265b8a25246c6611", "632c2f0fcde58668", "3213373733333726", "3723931253133343", "313c079b1c3e12c6", "b9ca0f369706acad", "1a29b565613a1b43", "fbe34f193b6ef074", "99690c96960d6923", "1223079cd8db9b9a", "d89d17578bca6899", "9346363322340cd8", "235398048f9be5ea", "192bca992d6cadfc", "b805367a7a4b5326", "b67462c1b1e3d8e2", "4b1b27a1d1e62e29", "8c07133b2f1f2ffd", "062626392d2d1b1a", "d4341a613133a84b", "660f0931d8cc66ac", "0a4f0e9f333ba695", "26172585b71a1b20", "b5397b5333b29212", "332e2638577b7193", "3399d972726a6272", "c4db31c58e2b326c", "4d3ab6a7a6dedbad"]]
print(len(TEST_DATOTEKE), 'datoteka i', len(TEST_HASH), 'otisaka testnih fotografija')

In [ ]:
# ----- 1. Pronađi svaku vrstu na iNaturalistu -----
S = requests.Session()
S.headers['User-Agent'] = 'GljiveHrvatske-trening/1.0 (github.com/markofrankovic-afk/Gljive-Hrvatske)'
API = 'https://api.inaturalist.org/v1/'

def api(path, **params):
    for pokusaj in range(6):
        try:
            r = S.get(API + path, params=params, timeout=60)
            time.sleep(API_PAUZA)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(30 * (pokusaj + 1)); continue
            print('  upit nije uspio', r.status_code, path, params); return None
        except requests.RequestException as e:
            print('  mreža:', e); time.sleep(20 * (pokusaj + 1))
    return None

TAKS_F = OUT / 'taksoni.json'
TAKS = json.loads(TAKS_F.read_text()) if TAKS_F.exists() else {}
RANKS = {'species', 'subspecies', 'variety', 'form', 'complex', 'hybrid'}
# tipfeleri i stari nazivi iz atlasa -> naziv pod kojim je vrsta na iNaturalistu
ISPRAVCI = {
 'Lactarius contorversus': 'Lactarius controversus', 'Hygrocybe intermedius': 'Hygrocybe intermedia',
 'Hygrocybe coccineus': 'Hygrocybe coccinea', 'Hygrocybe laeta': 'Gliophorus laetus', 'Camarophyllus niveus': 'Cuphophyllus virgineus',
 'Cantharellus cinerus': 'Craterellus cinereus', 'Boletus albidus': 'Caloboletus radicans', 'Xerocomus parasticus': 'Pseudoboletus parasiticus',
 'Leccinum criseum': 'Leccinum pseudoscabrum', 'Polyporus frandosus': 'Grifola frondosa', 'Polyporus intybaceus': 'Grifola frondosa',
 'Polyporus pes-caprae': 'Albatrellus pes-caprae', 'Calodon aurantiacum': 'Hydnellum aurantiacum', 'Calodon compactum': 'Hydnellum compactum',
 'Dryodon erinaceus': 'Hericium erinaceus', 'Pleurodon auriscalpium': 'Auriscalpium vulgare', 'Clavaria aurea': 'Ramaria aurea',
 'Clavaria pallida': 'Ramaria pallida', 'Clavaria fennica': 'Ramaria fennica', 'Clavaria pistillaris': 'Clavariadelphus pistillaris',
 'Clavaria truncata': 'Clavariadelphus truncatus', 'Tremellodon gelatinosum': 'Pseudohydnum gelatinosum', 'Exidia galndulosa': 'Exidia glandulosa',
 'Mitrophora hybrida': 'Morchella semilibera', 'Lepiota naucina': 'Leucoagaricus leucothites', 'Collybia platyphylla': 'Megacollybia platyphylla',
 'Dermocybe anthracina': 'Cortinarius anthracinus', 'Tricholoma georgi': 'Calocybe gambosa', 'Tricholoma tigrinum': 'Tricholoma pardinum',
 'Leucopaxillus amarus': 'Leucopaxillus gentianeus', 'Entoloma mougeoti': 'Entoloma mougeotii', 'Psathyrella spadiceo-grisea': 'Psathyrella spadiceogrisea',
 'Russula heterophila': 'Russula heterophylla', 'Amanita aspera': 'Amanita franchetii', 'Clitocybe gilva': 'Paralepista gilva',
 'Hygrophorus hypotheius': 'Hygrophorus hypothejus',
}
ATLAS_IMENA = {v.lower() for v in VRSTE} | {x.lower() for x in ISPRAVCI.values()}
SYN = {"boletus badius":"imleria badia","xerocomus badius":"imleria badia","boletus erythropus":"neoboletus erythropus","boletus luridiformis":"neoboletus erythropus","neoboletus luridiformis":"neoboletus erythropus","neoboletus praestigiator":"neoboletus erythropus","boletus luridus":"suillellus luridus","boletus queletii":"suillellus queletii","boletus satanas":"rubroboletus satanas","boletus calopus":"caloboletus calopus","boletus regius":"butyriboletus regius","boletus appendiculatus":"butyriboletus appendiculatus","boletus fuscoroseus":"butyriboletus pseudoregius","boletus pseudoregius":"butyriboletus pseudoregius","butyriboletus fuscoroseus":"butyriboletus pseudoregius","xerocomus chrysenteron":"xerocomellus chrysenteron","boletus chrysenteron":"xerocomellus chrysenteron","xerocomus rubellus":"hortiboletus rubellus","boletus rubellus":"hortiboletus rubellus","boletus subtomentosus":"xerocomus subtomentosus","leccinum crocipodium":"leccinellum crocipodium","leccinum carpini":"leccinum pseudoscabrum","leccinum griseum":"leccinum pseudoscabrum","lactarius piperatus":"lactifluus piperatus","lactarius volemus":"lactifluus volemus","lactarius vellereus":"lactifluus vellereus","clitocybe geotropa":"infundibulicybe geotropa","clitocybe gibba":"infundibulicybe gibba","clitocybe dealbata":"clitocybe rivulosa","collybia butyracea":"rhodocollybia butyracea","collybia fusipes":"gymnopus fusipes","oudemansiella radicata":"hymenopellis radicata","xerula radicata":"hymenopellis radicata","hygrocybe pratensis":"cuphophyllus pratensis","camarophyllus pratensis":"cuphophyllus pratensis","coprinus atramentarius":"coprinopsis atramentaria","coprinus picaceus":"coprinopsis picacea","lyophyllum connatum":"leucocybe connata","clitocybe connata":"leucocybe connata","agrocybe aegerita":"cyclocybe cylindracea","agrocybe cylindracea":"cyclocybe cylindracea","armillaria tabescens":"desarmillaria tabescens","lycoperdon pyriforme":"apioperdon pyriforme","langermannia gigantea":"calvatia gigantea","lepiota aspera":"echinoderma asperum","lepiota acutesquamosa":"echinoderma asperum","inocybe erubescens":"inosperma erubescens","inocybe patouillardii":"inosperma erubescens","inocybe rimosa":"pseudosperma rimosum","inocybe fastigiata":"pseudosperma rimosum","piptoporus betulinus":"fomitopsis betulina","volvariella gloiocephala":"volvopluteus gloiocephalus","volvariella speciosa":"volvopluteus gloiocephalus","paxillus atrotomentosus":"tapinella atrotomentosa","rozites caperatus":"cortinarius caperatus","rozites caperata":"cortinarius caperatus","cantharellus tubaeformis":"craterellus tubaeformis","cantharellus lutescens":"craterellus lutescens","macrolepiota rhacodes":"chlorophyllum rhacodes","macrolepiota rachodes":"chlorophyllum rhacodes","chlorophyllum rachodes":"chlorophyllum rhacodes","dendropolyporus umbellatus":"polyporus umbellatus","scutiger pes-caprae":"albatrellus pes-caprae","laeticutis cristata":"albatrellus cristatus","lepista saeva":"lepista personata","russula aurata":"russula aurea","hygrophoropsis aurantica":"hygrophoropsis aurantiaca","agaricus xanthoderma":"agaricus xanthodermus","agaricus haemorrhoidarius":"agaricus sylvaticus","gyromitra esculenta":"gyromitra esculenta","morchella conica":"morchella elata","hypholoma sublateritium":"hypholoma lateritium","psathyrella hydrophila":"psathyrella piluliformis","boletus reticulatus":"boletus reticulatus","boletus aestivalis":"boletus reticulatus","leccinellum pseudoscabrum":"leccinum pseudoscabrum","mucidula mucida":"oudemansiella mucida","russula heterophylla":"russula heterophila"}  # poznati parovi stari naziv -> novi naziv iz aplikacije
SYN = json.loads(SYN) if isinstance(SYN, str) else SYN
def smije_spojiti(q, ime):
    # spajanje s drugom vrstom iz atlasa samo ako su to poznati sinonimi (npr. Xerocomus badius = Imleria badia)
    q, ime = q.lower(), ime.lower()
    return ime not in ATLAS_IMENA - {q} or SYN.get(q) == ime or SYN.get(ime) == q
def _hit(r): return {'id': r['id'], 'name': r['name'], 'rank': r['rank'], 'obs': r.get('observations_count', 0)}
def _gljiva(r): return r.get('rank') in RANKS and r.get('iconic_taxon_name') in (None, 'Fungi', 'Protozoa')
def pogodak(j, q):
    res = [r for r in (j or {}).get('results', []) if _gljiva(r)]
    # 1. točno isti naziv ima prednost
    for r in res:
        if r.get('name', '').lower() == q.lower() and r.get('is_active', True) is not False:
            return _hit(r)
    # 2. sinonim, ali nikad spajanje s drugom vrstom koja u atlasu ima svoj opis
    for r in res:
        if (r.get('matched_term') or '').lower() == q.lower() and smije_spojiti(q, r.get('name', '')):
            return _hit(r)
    return None
def neaktivni(q):
    # stari naziv koji iNaturalist vodi kao neaktivan: prati ga do važećeg taksona
    j = api('taxa', q=q, is_active='any', per_page=30)
    for r in (j or {}).get('results', []):
        if r.get('name', '').lower() == q.lower() and r.get('is_active') is False:
            for tid in r.get('current_synonymous_taxon_ids') or []:
                t = (api(f'taxa/{tid}') or {}).get('results', [])
                if t and _gljiva(t[0]) and smije_spojiti(q, t[0].get('name', '')):
                    return _hit(t[0])
    return None
# v5: ponovno provjeri sve što nije točno isti naziv (stari popis je ponekad uzeo krivu vrstu)
if not TAKS.get('__v5__'):
    for v in VRSTE:
        t = TAKS.get(v)
        if t is False or (t and t['name'].lower() != ISPRAVCI.get(v, v).lower()):
            TAKS.pop(v, None)
# False = sigurno nema; None/nedostaje = upit nije uspio, pokušaj ponovno
nove = [v for v in VRSTE if TAKS.get(v) is None]
for i, ime in enumerate(nove):
    q = ISPRAVCI.get(ime, ime)
    j1 = api('taxa', q=q, per_page=30)
    hit = pogodak(j1, q)
    j2 = None
    if not hit:
        j2 = api('taxa/autocomplete', q=q, per_page=30)
        hit = pogodak(j2, q)
    if not hit and j1 is not None:
        hit = neaktivni(q)
    if not hit and j1 is not None:
        print('  nije pronađeno:', ime, '| iNaturalist nudi:', [r.get('name') + ' (' + str(r.get('rank')) + ')' for r in j1.get('results', [])[:3]])
    if hit:
        TAKS[ime] = hit
    elif j1 is not None and j2 is not None:
        TAKS[ime] = False
    if i % 25 == 0 or i == len(nove) - 1:
        TAKS_F.write_text(json.dumps(TAKS, ensure_ascii=False))
        print(f'{i+1}/{len(nove)}', ime, '->', hit and hit['name'])
TAKS['__v5__'] = True
TAKS_F.write_text(json.dumps(TAKS, ensure_ascii=False))
nema = [v for v in VRSTE if not TAKS.get(v)]
neuspjelo = [v for v in VRSTE if TAKS.get(v) is None]
if neuspjelo: print('⚠️ Za', len(neuspjelo), 'vrsta upit nije uspio; pokušat ću ponovno pri sljedećem pokretanju.')
print('Pronađeno', len(VRSTE) - len(nema), '/', len(VRSTE), '| nije pronađeno:', len(nema))
print(nema[:40])

In [ ]:
# ----- 2. Popis fotografija sa slobodnom licencom za svaku vrstu -----
def popis_za(ime, t):
    f = OUT / 'popis' / f"{t['id']}.json"
    if f.exists():
        c = json.loads(f.read_text())
        if isinstance(c, list): c = {'max': 300, 'fotke': c}   # popis iz v1/v2
        # ponovno koristi ako je popis dovoljno velik ili vrsta jednostavno nema više fotografija
        if c['max'] >= MAX_PO_VRSTI or len(c['fotke']) < c['max'] * 1.15:
            return c['fotke']
    fotke, page = [], 1
    while len(fotke) < MAX_PO_VRSTI * 1.15 and page <= 8:
        j = api('observations', taxon_id=t['id'], quality_grade='research', photos='true',
                photo_license=','.join(LICENCE), per_page=200, page=page, order_by='id', order='desc')
        if j is None:
            return None  # pokušaj opet pri sljedećem pokretanju
        res = j.get('results', [])
        for o in res:
            n = 0
            for p in o.get('photos', []):
                lic = (p.get('license_code') or '').lower()
                if lic not in LICENCE or not p.get('url'):
                    continue
                url = p['url'].replace('/square.', '/medium.')
                fotke.append({'pid': p['id'], 'oid': o['id'], 'ot': (o.get('taxon') or {}).get('id'),
                              'url': url, 'lic': lic, 'attr': p.get('attribution', ''),
                              'user': (o.get('user') or {}).get('login', '')})
                n += 1
                if n >= FOTO_PO_NALAZU: break
        if len(res) < 200: break
        page += 1
    f.write_text(json.dumps({'max': MAX_PO_VRSTI, 'fotke': fotke}, ensure_ascii=False))
    return fotke

POPIS = {}
lista = [(v, TAKS[v]) for v in VRSTE if TAKS.get(v)]
for i, (ime, t) in enumerate(lista):
    p = popis_za(ime, t)
    if p is not None:
        POPIS[ime] = p
    if i % 25 == 0 or i == len(lista) - 1:
        print(f'{i+1}/{len(lista)}', ime, len(p or []))
print('Ukupno fotografija:', sum(len(v) for v in POPIS.values()))

In [ ]:
# ----- 2b. Wikimedia Commons: dodatne fotografije (samo CC0, javno dobro i CC BY) -----
import re, html as _html
CAPI = 'https://commons.wikimedia.org/w/api.php'
CS = requests.Session(); CS.headers['User-Agent'] = S.headers['User-Agent']
(OUT / 'commons').mkdir(exist_ok=True)
NE = ('illustrat', 'drawing', 'herbar', 'microscop', 'micrograph', 'spore', 'stamp', 'painting', 'plate ', 'sketch',
      'exsiccat', 'diagram', 'map ', 'cyst', 'hyphae', 'basidi', 'ascus', 'asci', 'cheilocyst', 'specimen')

def capi(**params):
    params.update(format='json', formatversion=2)
    for pokusaj in range(5):
        try:
            r = CS.get(CAPI, params=params, timeout=60); time.sleep(API_PAUZA * 0.5)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503, 504): time.sleep(20 * (pokusaj + 1)); continue
            return None
        except requests.RequestException:
            time.sleep(10 * (pokusaj + 1))
    return None

def c_licenca(m):
    l = (m.get('License', {}).get('value') or '').lower().strip()
    kratko = (m.get('LicenseShortName', {}).get('value') or '').lower()
    if l.startswith('cc0') or 'cc0' in kratko: return 'cc0'
    if l == 'pd' or l.startswith('pd-') or kratko.startswith('public domain'): return 'pd'
    dijelovi = l.split('-')
    if l.startswith('cc-by-') and not any(x in dijelovi for x in ('sa', 'nc', 'nd')): return 'cc-by'
    return None

def c_tekst(m, k):
    return re.sub(r'<[^>]+>', '', _html.unescape(m.get(k, {}).get('value') or '')).strip()

def c_kategorija(kat, dubina=1):
    datoteke, podkat, cont = [], [], {}
    while True:
        j = capi(action='query', generator='categorymembers', gcmtitle='Category:' + kat, gcmtype='file|subcat',
                 gcmlimit=200, prop='imageinfo', iiprop='url|extmetadata|mime', iiurlwidth=500,
                 iiextmetadatafilter='License|LicenseShortName|Artist|Credit|ImageDescription', **cont)
        if j is None: return None
        for pg in (j.get('query') or {}).get('pages', []):
            t = pg.get('title', ''); tl = t.lower()
            if pg.get('ns') == 14:
                if not any(w in tl for w in NE): podkat.append(t.split(':', 1)[1])
                continue
            ii = (pg.get('imageinfo') or [{}])[0]; m = ii.get('extmetadata') or {}
            if ii.get('mime') != 'image/jpeg' or any(w in tl for w in NE) or t in TEST_DATOTEKE: continue
            if 'inaturalist' in (c_tekst(m, 'Credit') + c_tekst(m, 'ImageDescription')).lower(): continue  # kopije s iNaturalista već imamo
            lic = c_licenca(m)
            if not lic or not ii.get('thumburl'): continue
            autor = c_tekst(m, 'Artist')[:120] or 'Wikimedia Commons'
            datoteke.append({'pid': 'c%d' % pg['pageid'], 'oid': 'c%d' % pg['pageid'], 'ot': None, 'url': ii['thumburl'],
                             'lic': lic, 'attr': autor + ', ' + lic.upper() + ' (Wikimedia Commons)', 'user': autor,
                             'izvor': 'https://commons.wikimedia.org/wiki/' + t.replace(' ', '_')})
        if 'continue' not in j: break
        cont = {k: v for k, v in j['continue'].items()}
    if dubina > 0:
        for pk in podkat[:15]:
            d = c_kategorija(pk, dubina - 1)
            if d: datoteke += d
    return datoteke

def commons_za(ime):
    f = OUT / 'commons' / (hashlib.md5(ime.encode()).hexdigest() + '.json')
    if f.exists(): return json.loads(f.read_text())
    nazivi = [ime] + ([TAKS[ime]['name']] if TAKS.get(ime) and TAKS[ime]['name'] != ime else [])
    sve, vidjeno = [], set()
    for n in nazivi:
        d = c_kategorija(n)
        if d is None: return None   # ponovno pri sljedećem pokretanju
        for x in d:
            if x['pid'] not in vidjeno: vidjeno.add(x['pid']); sve.append(x)
    f.write_text(json.dumps(sve, ensure_ascii=False))
    return sve

COMMONS_FOTO = {}
if COMMONS:
    for i, ime in enumerate(VRSTE):
        d = commons_za(ime)
        if d is not None: COMMONS_FOTO[ime] = d
        if i % 50 == 0 or i == len(VRSTE) - 1:
            print(f'{i+1}/{len(VRSTE)}', ime, len(d or []))
    print('Commons: vrsta s fotografijama', sum(1 for v in COMMONS_FOTO.values() if v), '| fotografija', sum(len(v) for v in COMMONS_FOTO.values()))

In [ ]:
# ----- 3. Razredi: spoji sinonime, makni duplikate, preskoči vrste s premalo fotografija -----
po_taks = {}
for ime in VRSTE:
    t = TAKS.get(ime)
    kljuc = t['id'] if t else 'n:' + ime.lower()
    if (t and ime in POPIS) or COMMONS_FOTO.get(ime):
        po_taks.setdefault(kljuc, []).append(ime)
vlasnik = {}   # pid -> taksonski id razreda (precizniji takson ima prednost)
for tid, imena in po_taks.items():
    for p in POPIS.get(imena[0], []):
        if p['pid'] not in vlasnik or p['ot'] == tid:
            vlasnik[p['pid']] = tid
c_vlasnik = {}  # ista datoteka s Commonsa samo u jednom razredu
for tid, imena in po_taks.items():
    for ime in imena:
        for p in COMMONS_FOTO.get(ime, []):
            c_vlasnik.setdefault(p['pid'], tid)
RAZREDI, PRESKOCENE = [], []
for tid, imena in po_taks.items():
    fot = [p for p in POPIS.get(imena[0], []) if vlasnik.get(p['pid']) == tid][:MAX_PO_VRSTI]
    cf, vid = [], set()
    for ime in imena:
        for p in COMMONS_FOTO.get(ime, []):
            if c_vlasnik.get(p['pid']) == tid and p['pid'] not in vid: vid.add(p['pid']); cf.append(p)
    fot = fot + cf[:COMMONS_MAX]
    if len(fot) >= MIN_PO_VRSTI:
        RAZREDI.append({'ime': imena[0], 'sinonimi': imena[1:], 'tid': tid, 'fotke': fot})
    else:
        PRESKOCENE.append((imena[0], len(fot)))
RAZREDI.sort(key=lambda r: r['ime'])
print('Razreda za učenje:', len(RAZREDI), '| preskočeno (premalo slobodnih fotografija):', len(PRESKOCENE))
print('Fotografija:', sum(len(r['fotke']) for r in RAZREDI), '| od toga s Commonsa:', sum(1 for r in RAZREDI for p in r['fotke'] if str(p['pid']).startswith('c')))

In [ ]:
# ----- 4. Skidanje fotografija (spremaju se smanjene, kratka strana 288 px) -----
TAR = OUT / 'slike.tar'
if TAR.exists() and not any(SLIKE.iterdir()):
    print('Raspakiravam ranije skinute fotografije s Drivea…')
    with tarfile.open(TAR) as t:
        try: t.extractall(WORK, filter='data')
        except TypeError: t.extractall(WORK)

def skini(p):
    f = SLIKE / f"{p['pid']}.jpg"
    if f.exists():
        return True
    for pokusaj in range(3):
        try:
            r = requests.get(p['url'], timeout=30)
            if r.status_code != 200:
                time.sleep(2); continue
            im = Image.open(io.BytesIO(r.content)).convert('RGB')
            k = 288 / min(im.size)
            if k < 1:
                im = im.resize((max(1, round(im.width * k)), max(1, round(im.height * k))), Image.BICUBIC)
            im.save(f, 'JPEG', quality=88)
            return True
        except Exception:
            time.sleep(2)
    return False

sve = [p for r in RAZREDI for p in r['fotke']]
treba = [p for p in sve if not (SLIKE / f"{p['pid']}.jpg").exists()]
print('Treba skinuti', len(treba), 'od', len(sve))
ok = 0
t_inat = [p for p in treba if not str(p['pid']).startswith('c')]
t_comm = [p for p in treba if str(p['pid']).startswith('c')]
for skup, niti in ((t_inat, 16), (t_comm, 4)):   # Wikimedia traži umjeren broj istovremenih zahtjeva
    with ThreadPoolExecutor(niti) as ex:
        for i, res in enumerate(ex.map(skini, skup)):
            ok += res
            if i % 2000 == 0: print(f'  {i}/{len(skup)}')
print('Skinuto', ok, '| nije uspjelo', len(treba) - ok)
if treba and ok:
    print('Spremam kopiju fotografija na Drive (da sljedeći put ne treba skidati)…')
    with tarfile.open(TAR, 'w') as t: t.add(SLIKE, arcname='slike')

# fotografije koje su (gotovo) iste kao testne izbacuju se iz učenja
def otisak(f):
    try:
        im = Image.open(f).convert('L').resize((9, 8), Image.LANCZOS); px = im.tobytes(); v = 0
        for r in range(8):
            for c in range(8): v = (v << 1) | (px[r * 9 + c] > px[r * 9 + c + 1])
        return v
    except Exception:
        return None
OTISCI_F = OUT / 'otisci.json'
OTISCI = json.loads(OTISCI_F.read_text()) if OTISCI_F.exists() else {}
nove_o = [p for p in sve if str(p['pid']) not in OTISCI and (SLIKE / f"{p['pid']}.jpg").exists()]
with ThreadPoolExecutor(8) as ex:
    for p, o in zip(nove_o, ex.map(lambda p: otisak(SLIKE / f"{p['pid']}.jpg"), nove_o)):
        OTISCI[str(p['pid'])] = o
OTISCI_F.write_text(json.dumps(OTISCI))
ISTE_KAO_TEST = {pid for pid, o in OTISCI.items() if o is not None and min(bin(o ^ h).count('1') for h in TEST_HASH) <= 4}
print('Izbačeno jer su iste kao testne fotografije:', len(ISTE_KAO_TEST))

In [ ]:
# ----- 5. Podjela na učenje i provjeru (10 % nalaza ide u provjeru, po nalazu da nema prepisivanja) -----
def u_provjeri(oid):
    return int(hashlib.md5(str(oid).encode()).hexdigest(), 16) % 10 == 0
TRAIN, VAL = [], []
for ci, r in enumerate(RAZREDI):
    for p in r['fotke']:
        f = SLIKE / f"{p['pid']}.jpg"
        if f.exists() and str(p['pid']) not in ISTE_KAO_TEST:
            (VAL if u_provjeri(p['oid']) else TRAIN).append((str(f), ci))
LABELS = [r['ime'] for r in RAZREDI]
print('Učenje:', len(TRAIN), '| provjera:', len(VAL), '| razreda:', len(LABELS))

In [ ]:
# ----- 6. Treniranje -----
import timm
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=len(LABELS))
cfg = model.pretrained_cfg
MEAN, STD = list(cfg.get('mean', (0.485, 0.456, 0.406))), list(cfg.get('std', (0.229, 0.224, 0.225)))

class DS(Dataset):
    def __init__(s, items, tf): s.items, s.tf = items, tf
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        f, c = s.items[i]
        try: im = Image.open(f).convert('RGB')
        except Exception: im = Image.new('RGB', (IMG, IMG))
        return s.tf(im), c

NW = min(8, os.cpu_count() or 2)
# rijetke vrste dolaze malo češće na red (težina 1/korijen broja fotografija)
from torch.utils.data import WeightedRandomSampler
from collections import Counter
broj = Counter(c for _, c in TRAIN)
sampler = WeightedRandomSampler([1 / math.sqrt(broj[c]) for _, c in TRAIN], num_samples=len(TRAIN), replacement=True)

def faza(zavrsna):
    # osnovna faza uči na IMG px; zadnje epohe na IMG_FINAL px, a provjera i aplikacija gledaju cijelu sliku na IMG_FINAL
    global tr_val, dl_tr, dl_va, IMG_NOW, CROP
    IMG_NOW, CROP = (IMG_FINAL, 1.0) if zavrsna else (IMG, 0.875)
    tr_train = T.Compose([T.RandomResizedCrop(IMG_NOW, scale=(0.35 if zavrsna else 0.3, 1.0), interpolation=T.InterpolationMode.BICUBIC),
                          T.RandomHorizontalFlip(), T.ColorJitter(0.25, 0.25, 0.15, 0.0), T.ToTensor(), T.Normalize(MEAN, STD)])
    tr_val = T.Compose([T.Resize(round(IMG_NOW / CROP), interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(IMG_NOW),
                        T.ToTensor(), T.Normalize(MEAN, STD)])
    dl_tr = DataLoader(DS(TRAIN, tr_train), batch_size=BATCH, sampler=sampler, num_workers=NW, pin_memory=True, drop_last=True, persistent_workers=NW > 0)
    dl_va = DataLoader(DS(VAL, tr_val), batch_size=BATCH, shuffle=False, num_workers=NW, pin_memory=True)
ZAVRSNA = lambda ep: FINAL_EP > 0 and ep >= EPOHE - FINAL_EP
faza(False)

model = model.to(DEV).to(memory_format=torch.channels_last)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.02)
steps = EPOHE * len(dl_tr); warm = len(dl_tr)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
scaler = torch.amp.GradScaler('cuda', enabled=DEV == 'cuda')
crit = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

CKPT, BEST = OUT / f'ckpt_v{VERZIJA}.pt', OUT / f'najbolji_v{VERZIJA}.pt'
pocetak, najbolji = 0, 0.0
if CKPT.exists():
    ck = torch.load(CKPT, map_location=DEV, weights_only=False)
    if ck.get('labels') == LABELS and ck.get('model_name') == MODEL_NAME:
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        scaler.load_state_dict(ck['scaler']); pocetak, najbolji = ck['epoha'] + 1, ck['najbolji']
        print('Nastavljam od epohe', pocetak + 1)
    else:
        print('Popis vrsta se promijenio, treniram ispočetka.')

def provjera():
    model.eval(); t1 = t3 = n = 0
    with torch.no_grad(), torch.autocast(DEV, enabled=DEV == 'cuda'):
        for x, y in dl_va:
            x, y = x.to(DEV).to(memory_format=torch.channels_last), y.to(DEV)
            top = model(x).topk(3, 1).indices
            t1 += (top[:, 0] == y).sum().item(); t3 += (top == y[:, None]).any(1).sum().item(); n += len(y)
    model.train(); return t1 / max(1, n), t3 / max(1, n)

if ZAVRSNA(pocetak): faza(True)
for ep in range(pocetak, EPOHE):
    if ZAVRSNA(ep) and IMG_NOW != IMG_FINAL:
        faza(True)
    if ZAVRSNA(ep) and ep == EPOHE - FINAL_EP:
        najbolji = 0.0; print(f'Završna faza: učim na {IMG_FINAL} px')
    model.train(); t0 = time.time(); zbroj = 0.0
    for i, (x, y) in enumerate(dl_tr):
        x, y = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last), y.to(DEV, non_blocking=True)
        with torch.autocast(DEV, enabled=DEV == 'cuda'):
            loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        zbroj += loss.item()
        if i % 200 == 0: print(f'  epoha {ep+1} korak {i}/{len(dl_tr)} gubitak {loss.item():.3f}')
    a1, a3 = provjera()
    print(f'Epoha {ep+1}/{EPOHE}: gubitak {zbroj/max(1,len(dl_tr)):.3f} | točno {a1*100:.1f} % | među prva 3: {a3*100:.1f} % | {(time.time()-t0)/60:.1f} min')
    if a1 >= najbolji:
        najbolji = a1; torch.save({'model': model.state_dict(), 'labels': LABELS, 'top1': a1, 'top3': a3, 'img': IMG_NOW, 'crop': CROP}, BEST)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                'epoha': ep, 'najbolji': najbolji, 'labels': LABELS, 'model_name': MODEL_NAME}, CKPT)
print('Najbolja točnost na provjeri:', round(najbolji * 100, 1), '%')

In [ ]:
# ----- 7. Izvoz za aplikaciju (ONNX) i paket za Claudea -----
import onnx, onnxruntime as ort, numpy as np
best = torch.load(BEST, map_location='cpu', weights_only=False)
m = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABELS)); m.load_state_dict(best['model']); m.eval()
PAKET = OUT / 'paket'; shutil.rmtree(PAKET, ignore_errors=True); PAKET.mkdir()
IMG_E, CROP_E = best.get('img', IMG), best.get('crop', 0.875)
tr_val = T.Compose([T.Resize(round(IMG_E / CROP_E), interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(IMG_E), T.ToTensor(), T.Normalize(MEAN, STD)])
onx = PAKET / 'model.onnx'; x = torch.randn(1, 3, IMG_E, IMG_E)
kw = dict(input_names=['input'], output_names=['logits'], opset_version=17, do_constant_folding=True)
try:
    torch.onnx.export(m, x, str(onx), dynamo=False, **kw)
except TypeError:
    torch.onnx.export(m, x, str(onx), **kw)
onnx.checker.check_model(str(onx))
xr = torch.stack([tr_val(Image.open(f).convert('RGB')) for f, _ in VAL[:8]]) if VAL else x
o_onnx = np.concatenate([ort.InferenceSession(str(onx)).run(None, {'input': xr[i:i+1].numpy()})[0] for i in range(len(xr))])
with torch.no_grad(): o_pt = m(xr).numpy()
razlika = float(np.abs(o_onnx - o_pt).max() / (np.abs(o_pt).max() + 1e-6))
print('ONNX provjera: relativna razlika', round(razlika, 6), '| isti prvi prijedlog:', bool((o_onnx.argmax(1) == o_pt.argmax(1)).all()))
assert razlika < 1e-3 and (o_onnx.argmax(1) == o_pt.argmax(1)).all()

data = onx.read_bytes(); onx.unlink(); DIO = 12_000_000; parts, sizes = [], []
for k in range(0, len(data), DIO):
    n = f'model.onnx.{k // DIO:03d}'; (PAKET / n).write_bytes(data[k:k + DIO]); parts.append(n); sizes.append(len(data[k:k + DIO]))
meta = {'model': f'gh-{MODEL_NAME}-v{VERZIJA}', 'input_size': [3, IMG_E, IMG_E], 'mean': MEAN, 'std': STD, 'crop_pct': CROP_E,
        'interpolation': 'bicubic', 'num_classes': len(LABELS), 'verzija': VERZIJA, 'vlastiti': True,
        'licencija': f'Vlastiti model Gljiva Hrvatske. Osnova: timm {MODEL_NAME} (Apache 2.0). Fotografije: iNaturalist i Wikimedia Commons, CC0, javno dobro i CC BY (autori u zasluge.csv).',
        'top1_provjera': round(best['top1'], 4), 'top3_provjera': round(best['top3'], 4), 'parts': parts, 'part_sizes': sizes}
(PAKET / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=1))
(PAKET / 'labels.json').write_text(json.dumps(LABELS, ensure_ascii=False))
with open(PAKET / 'zasluge.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['vrsta', 'foto_id', 'nalaz', 'autor', 'licenca', 'navod'])
    for r in RAZREDI:
        for p in r['fotke']:
            if (SLIKE / f"{p['pid']}.jpg").exists() and str(p['pid']) not in ISTE_KAO_TEST:
                w.writerow([r['ime'], p['pid'], p.get('izvor') or f"https://www.inaturalist.org/observations/{p['oid']}", p['user'], p['lic'], p['attr']])
(PAKET / 'izvjestaj.json').write_text(json.dumps({
    'razreda': len(LABELS), 'ucenje': len(TRAIN), 'provjera': len(VAL), 'top1': best['top1'], 'top3': best['top3'],
    'sinonimi': {r['ime']: r['sinonimi'] for r in RAZREDI if r['sinonimi']},
    'commons_po_vrsti': {r['ime']: sum(1 for p in r['fotke'] if str(p['pid']).startswith('c')) for r in RAZREDI},
    'iste_kao_test': len(ISTE_KAO_TEST),
    'preskocene': PRESKOCENE, 'nije_pronadeno': [v for v in VRSTE if not TAKS.get(v)]}, ensure_ascii=False, indent=1))
ZIP = OUT / f'GljiveModel_v{VERZIJA}.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(PAKET.iterdir()): z.write(f, f.name)
print('✅ Gotovo:', ZIP, round(ZIP.stat().st_size / 1e6, 1), 'MB')
print('Pošalji Claudeu datoteku GljiveModel_v%d.zip iz mape GljiveModel na Google Driveu.' % VERZIJA)
try:
    from google.colab import files; files.download(str(ZIP))
except Exception:
    pass